In [0]:
from pyspark.sql import functions as F

# 1. Caminhos
path_silver = "dbfs:/Volumes/workspace/default/data/delta/silver/"
path_gold = "dbfs:/Volumes/workspace/default/data/delta/gold/"

# 2. Carregar e tratar dados (Blindagem contra erros de cast)
orders_consolidated = spark.read.format("delta").load(f"{path_silver}orders_consolidated") \
    .select(
        "seller_id", 
        "seller_city", 
        "seller_state", 
        "order_id",
        F.expr("try_cast(price as double)").alias("price")
    )

reviews = spark.read.format("delta").load("dbfs:/Volumes/workspace/default/data/delta/bronze/reviews") \
    .select(
        "order_id", 
        F.expr("try_cast(review_score as double)").alias("review_score")
    )

# 3. Média de review por pedido
reviews_avg_per_order = reviews.groupBy("order_id").agg(
    F.avg("review_score").alias("avg_review_score")
)

# 4. Join e Agregação Final
gold_seller_summary = orders_consolidated.join(reviews_avg_per_order, "order_id", "left") \
    .groupBy("seller_id", "seller_city", "seller_state") \
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("price").alias("total_revenue"),
        F.avg("avg_review_score").alias("avg_review_score")
    )

# 5. Salvamento com a opção de Sobrescrever o Esquema (CORREÇÃO DO ERRO)
gold_seller_summary.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{path_gold}seller_summary")

print("✨ Tabela 'seller_summary' salva com sucesso e esquema atualizado!")

✨ Tabela 'seller_summary' salva com sucesso e esquema  atualizado!
